<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/HOGENT-ML/course/blob/main/720-exercise_churn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
</table>

In [1]:
# Importing the necessary packages
import numpy as np                                  # "Scientific computing"
import scipy.stats as stats                         # Statistical tests

import pandas as pd                                 # Data Frame
from pandas.api.types import CategoricalDtype

import matplotlib.pyplot as plt                     # Basic visualisation

from sklearn.model_selection import cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler

Churn prediction is one of the classic machine learning applications. Companies want to predict the likelihood of a customer or employee leaving. Customers or employees that are "in danger" can then get a special treatment. The dataset we use in this exercise contains historical data from bank customers. We know for each customer wether he/she left ("Exited") or not. 

In [2]:
#churn = pd.read_csv('https://raw.githubusercontent.com/HOGENT-ML/course/main/datasets/churn.csv')
churn = pd.read_csv('datasets/churn.csv')
churn.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


Get some general info about the dataset (type of each column, null values, ...)

In [3]:
churn.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  str    
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  str    
 5   Gender           10000 non-null  str    
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), str(3)
memory usage: 1.1 MB


Perform basic data cleaning and preparation. 

Tip: use the solution of the exercise "demographic student score" as a source of inspiration. 

Remove the columns you don't need

In [4]:
churn.drop(['RowNumber', 'CustomerId','Surname'], axis=1, inplace=True)
churn.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CreditScore      10000 non-null  int64  
 1   Geography        10000 non-null  str    
 2   Gender           10000 non-null  str    
 3   Age              10000 non-null  int64  
 4   Tenure           10000 non-null  int64  
 5   Balance          10000 non-null  float64
 6   NumOfProducts    10000 non-null  int64  
 7   HasCrCard        10000 non-null  int64  
 8   IsActiveMember   10000 non-null  int64  
 9   EstimatedSalary  10000 non-null  float64
 10  Exited           10000 non-null  int64  
dtypes: float64(2), int64(7), str(2)
memory usage: 859.5 KB


Is this a skewed dataset?

In [5]:
churn['Exited'].value_counts()

Exited
0    7963
1    2037
Name: count, dtype: int64

What is X and what is y?

In [6]:
X = churn.drop(['Exited'], axis=1)
y = churn['Exited']

Define the data preparation for the categorical and numerical columns
Setting remainder='passthrough' will mean that all columns not specified in the list of "transformers"  
will be passed through without transformation, instead of being dropped.

In [7]:
from sklearn.preprocessing import OneHotEncoder

numerical_cols = X.select_dtypes(include=['number']).columns
categorical_cols = X.select_dtypes(include=['str', 'object']).columns

#num_tr = ColumnTransformer([('scale', MinMaxScaler(), numerical_cols), ()], remainder='passthrough')
#cat_tr = ColumnTransformer([(), ()], remainder='passthrough')

col_transform = ColumnTransformer(transformers=[('cat', OneHotEncoder(sparse_output=False), categorical_cols), ('num', MinMaxScaler(), numerical_cols)], remainder='passthrough')

What is X_train, y_train, X_test, y_test?

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

Find a model for LogisticRegression, Support Vector Machines with 3d degree polynomial kernel, Decision Trees and Random Forest each with their default parameters. Which one gives the best accuracy?

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

classifiers = [('lr', LogisticRegression(random_state=42)),
               ('svc_poly', SVC(random_state=42, kernel='poly', degree=3)),
               ('tree', DecisionTreeClassifier(random_state=42)),
               ('rf', RandomForestClassifier(random_state=42))]

for name,model in classifiers:
    pipeline = Pipeline(steps=[('prep', col_transform), (name, model)])
    acc = np.mean(cross_val_score(pipeline, X_train, y_train, cv=3, scoring='accuracy'))
    print(f'accuracy of model {name}: {acc:.3}')

accuracy of model lr: 0.812
accuracy of model svc_poly: 0.85
accuracy of model tree: 0.791
accuracy of model rf: 0.859


Does a soft voting classifier using the above classifiers perform better?

In [10]:
from sklearn.ensemble import VotingClassifier, StackingClassifier

voting_cfl_hard = VotingClassifier(estimators=classifiers, voting='hard')
voting_cfl_soft = VotingClassifier(estimators=classifiers, voting='soft')
voting_cfl_soft.named_estimators['svc_poly'].probability = True
pipeline_soft = Pipeline(steps=[('prep', col_transform), ('voting_soft', voting_cfl_soft)])
pipeline_hard = Pipeline(steps=[('prep', col_transform), ('voting_hard', voting_cfl_hard)])
pipeline_soft.fit(X_train, y_train)
pipeline_hard.fit(X_train, y_train)
acc_soft = np.mean(cross_val_score(pipeline_soft, X_train, y_train, cv=3, scoring='accuracy'))
acc_hard = np.mean(cross_val_score(pipeline_hard, X_train, y_train, cv=3, scoring='accuracy'))

print(f'hard voting accuracy: {acc_hard}')
print(f'soft voting accuracy: {acc_soft}')

hard voting accuracy: 0.8466666666666667
soft voting accuracy: 0.8513333333333333


Continue with the best model from  the 4 individual classifiers above and apply grid search to find the best parameter combination. 

What's the best parameter combination and the corresponding accuracy?

In [11]:
from sklearn.model_selection import GridSearchCV

pipeline = Pipeline([('preprocessing', col_transform),
                     ('rf', RandomForestClassifier(random_state=42))])

param_grid = [
                {
                'rf__n_estimators': [30, 50, 100, 200], 
                'rf__bootstrap': [True, False], 
                'rf__max_features': [4, 6, 8, 10]
                }
             ]

grid_search = GridSearchCV(pipeline, param_grid=param_grid, cv=3, scoring='accuracy')
grid_search.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'rf__bootstrap': [True, False], 'rf__max_features': [4, 6, ...], 'rf__n_estimators': [30, 50, ...]}]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is

In [12]:
grid_search.best_params_

{'rf__bootstrap': True, 'rf__max_features': 6, 'rf__n_estimators': 100}

In [13]:
grid_search.best_score_

np.float64(0.8597333333333333)

What is the accuracy score on the test set and what are the most important features?

In [14]:
from sklearn.metrics import accuracy_score

rf = RandomForestClassifier(max_features=6, n_estimators=100, random_state=42)
pipeline = Pipeline(steps=[('prep', col_transform), ('rf', rf)])
pipeline.fit(X_train, y_train)
acc = pipeline.score(X_test, y_test)

print(f'Accuracy: {acc}')

Accuracy: 0.8664


In [15]:
rf.feature_importances_

array([0.01032435, 0.02225888, 0.01016472, 0.0124365 , 0.01228353,
       0.13876004, 0.23042173, 0.07454065, 0.14601422, 0.13077118,
       0.01885067, 0.04950777, 0.14366576])

In [16]:
col_transform.get_feature_names_out()

array(['cat__Geography_France', 'cat__Geography_Germany',
       'cat__Geography_Spain', 'cat__Gender_Female', 'cat__Gender_Male',
       'num__CreditScore', 'num__Age', 'num__Tenure', 'num__Balance',
       'num__NumOfProducts', 'num__HasCrCard', 'num__IsActiveMember',
       'num__EstimatedSalary'], dtype=object)

In [17]:
for score, name in zip(rf.feature_importances_, col_transform.get_feature_names_out()):
    print(round(score, 2), name)

0.01 cat__Geography_France
0.02 cat__Geography_Germany
0.01 cat__Geography_Spain
0.01 cat__Gender_Female
0.01 cat__Gender_Male
0.14 num__CreditScore
0.23 num__Age
0.07 num__Tenure
0.15 num__Balance
0.13 num__NumOfProducts
0.02 num__HasCrCard
0.05 num__IsActiveMember
0.14 num__EstimatedSalary


Do Ada Boosting or Stacking lead to a better accuracy? 

For Stacking you can use the same estimators as you did for voting, but apply for the best classifier the optimal parameter combination you found above. 

In [18]:
from sklearn.ensemble import AdaBoostClassifier

X_train_prep = col_transform.fit_transform(X_train)
X_test_prep = col_transform.transform(X_test)

ada_clf = AdaBoostClassifier(RandomForestClassifier(n_estimators=100, max_features=4, random_state=42), learning_rate=0.5, random_state=42)
ada_clf.fit(X_train_prep, y_train)
print(ada_clf.score(X_test_prep, y_test))

0.8692


In [20]:
from sklearn.ensemble import StackingClassifier

classifiers = [('lr', LogisticRegression(random_state=42)),
               ('svc_poly', SVC(random_state=42, kernel='poly', degree=3)),
               ('tree', DecisionTreeClassifier(random_state=42)),
               ('rf', RandomForestClassifier(n_estimators=100, max_features=6, random_state=42))]

stacking_clf = StackingClassifier(estimators=classifiers, final_estimator=RandomForestClassifier(n_estimators=100, max_features=4, random_state=42), cv=5)
stacking_clf.fit(X_train_prep, y_train)
print(stacking_clf.score(X_test_prep, y_test))

0.85


Conclusion: which model delivers the best results? 